# DIRE trên Kaggle — smoke test (miễn phí, 0 credit Modal)

Notebook này là **bước 1** của `docs/runbook-free.md`: kiểm tra code chạy đúng trên mẫu nhỏ
trước khi tiêu credit Modal vào việc train thật.

**Kaggle cho gì:** ~30 giờ GPU/tuần (P100 16 GB), phiên ~9–12 giờ, **không SSH**, **không chạy nền**, idle 60 phút là mất máy.
→ Chỉ dùng để *dev*: sửa code, chạy 1–2k ảnh, kiểm tra logic. Train thật chạy trên Modal.

**Cần chuẩn bị (1 lần):**
1. Upload mirror dữ liệu lên Kaggle Datasets (xem `docs/runbook-free.md` bước 5) **hoặc** dùng đường tải mẫu nhỏ từ Hugging Face trong cell 3b.
2. Trong Kaggle: *Settings → Accelerator → GPU P100*.

In [ ]:
# 1. Kiểm tra môi trường
import os, sys, subprocess, glob, json

print('python  :', sys.version.split()[0])
print('disk    :', subprocess.run(['df', '-h', '/kaggle/working'], capture_output=True, text=True).stdout.splitlines()[-1])
try:
    import torch
    print('torch   :', torch.__version__)
    print('cuda    :', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
except ImportError:
    print('torch   : chưa có (cài ở cell sau)')

print('\n/kaggle/input (dataset đã mount):')
for p in sorted(glob.glob('/kaggle/input/*'))[:20]:
    print('  -', p)

In [ ]:
# 2. Cài phụ thuộc + lấy code repo
# Repo riêng tư → upload file zip vào Kaggle Dataset, hoặc pip từ git nếu bạn để public.
import subprocess, sys, glob, os

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'webdataset', 'pillow', 'pyyaml', 'boto3'], check=True)

REPO_ZIP = glob.glob('/kaggle/input/*/dire-modal*.zip') + glob.glob('/kaggle/input/*/*/dire-modal*.zip')
if REPO_ZIP:
    subprocess.run(['unzip', '-q', '-o', REPO_ZIP[0], '-d', '/kaggle/working/repo'], check=True)
    os.chdir('/kaggle/working/repo/dire-modal' if os.path.isdir('/kaggle/working/repo/dire-modal') else '/kaggle/working/repo')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)
    print('cài từ zip:', os.getcwd())
else:
    # Nếu repo public (không có secret nào trong đó):
    GIT = 'https://github.com/jackboom2510/dire-modal'
    print(f'không thấy zip trong /kaggle/input — thử pip từ git: {GIT}')
    try:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', f'git+{GIT}'], check=True)
    except subprocess.CalledProcessError as exc:
        print('⚠ Không cài được (repo private?). Hãy upload zip vào Kaggle Dataset:', exc)

sys.path.insert(0, os.getcwd())

In [ ]:
# 3. Tìm dữ liệu DIRE
# Ưu tiên (a) mirror trên Kaggle Datasets đã mount, (b) tải mẫu nhỏ từ HF.
import glob, os

candidates = glob.glob('/kaggle/input/*/dire/**/*.tar.gz', recursive=True)
if candidates:
    DATA_ROOT = '/kaggle/input'
    print(f'[a] thấy {len(candidates)} archive trong /kaggle/input')
    for c in candidates[:10]:
        print('   ', c, f'{os.path.getsize(c)/1e6:.1f} MB')
else:
    print('[b] không thấy mirror → sẽ tải mẫu nhỏ (nếu có HF_REPO_ID)')
    DATA_ROOT = '/kaggle/working/sample'
    HF_REPO_ID = os.environ.get('HF_REPO_ID', '')  # đặt qua Kaggle Secrets nếu muốn
    if HF_REPO_ID:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub'], check=True)
        from huggingface_hub import snapshot_download
        snapshot_download(repo_id=HF_REPO_ID, repo_type='dataset', local_dir=DATA_ROOT,
                          allow_patterns=['*sample*', '*manifest*'])
        print('đã tải mẫu →', DATA_ROOT)
    else:
        print('⚠ Chưa cấu hình dữ liệu. Xem docs/runbook-free.md bước 5 để tạo mirror.')
print('\nDATA_ROOT =', DATA_ROOT)

In [ ]:
# 4. Pack 1 archive nhỏ → WebDataset shard (kiểm tra logic pack.py chạy được)
import tarfile, io, os, json
from pathlib import Path

OUT = Path('/kaggle/working/shards'); OUT.mkdir(exist_ok=True)

archives = sorted(Path(DATA_ROOT).rglob('*.tar.gz'))
if not archives:
    print('⚠ Không có archive nào — bỏ qua cell này (xem bước 5 của runbook)')
else:
    src = min(archives, key=lambda p: p.stat().st_size)  # nhỏ nhất để chạy nhanh
    print(f'pack {src.name} ({src.stat().st_size/1e6:.1f} MB)')
    import webdataset as wds
    out_dir = OUT / src.name.replace('.tar.gz', ''); out_dir.mkdir(parents=True, exist_ok=True)
    label = 0 if '/real/' in str(src) or src.name.startswith('real') else 1
    n = 0
    with wds.ShardWriter(str(out_dir / 'shard-%06d.tar'), maxcount=500) as sink, tarfile.open(src, 'r:gz') as tf:
        for m in tf:
            if not m.isfile():
                continue
            name = m.name.lower()
            if not name.endswith(('.png', '.jpg', '.jpeg')):
                continue
            sink.write({'__key__': f'{src.stem}/{n:08d}', name.rsplit('.', 1)[-1]: tf.extractfile(m).read(), 'cls': str(label).encode()})
            n += 1
    (out_dir / 'meta.json').write_text(json.dumps({'src': str(src), 'count': n, 'label': label}, indent=2))
    print(f'[ok] {n} ảnh → {out_dir}')

In [ ]:
# 5. DataLoader + 1 bước train giả (kiểm tra đường đi dữ liệu end-to-end)
import torch, torch.nn as nn, torchvision.transforms as T
from PIL import Image
import webdataset as wds, glob

shards = sorted(glob.glob('/kaggle/working/shards/**/*.tar', recursive=True))
if not shards:
    print('⚠ Không có shard — chạy cell 4 trước (hoặc bỏ qua nếu chưa có dữ liệu)')
else:
    tf = T.Compose([T.Resize(256), T.CenterCrop(256), T.ToTensor()])

    def decode(sample):
        img = sample.get('png') or sample.get('jpg') or sample.get('jpeg')
        x = tf(Image.open(__import__('io').BytesIO(img)).convert('RGB'))
        y = int(sample['cls'].decode() if isinstance(sample['cls'], bytes) else sample['cls'])
        return x, y

    ds = wds.WebDataset(shards).decode('rgb').map(decode).batched(16)
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = torchvision.models.resnet18(num_classes=2) if False else None
    import torchvision
    model = torchvision.models.resnet18(num_classes=2).to(dev)
    opt = torch.optim.SGD(model.parameters(), lr=1e-3)
    crit = nn.CrossEntropyLoss()

    steps = 0
    for x, y in ds:
        x, y = x.to(dev), y.to(dev)
        loss = crit(model(x), y)
        opt.zero_grad(); loss.backward(); opt.step()
        steps += 1
        print(f'step {steps}: loss={loss.item():.4f} batch={tuple(x.shape)} device={dev}')
        if steps >= 3:
            break
    print(f'\n[ok] đường đi dữ liệu hoạt động — {steps} bước trên {dev}')
    print('→ Giờ có thể tin code trước khi tiêu credit Modal để train thật.')

## Bước tiếp theo

Sau khi notebook này chạy sạch:

1. **Train thật trên Modal** (~3–8 A100-giờ ≈ $8–20 trong $30 credit):
   ```bash
   ./scripts/05_train_repro.sh
   python3 tools/free_quota.py --used modal=0
   ```
2. **Eval 16 generator** song song: `./scripts/04_eval_pretrained.sh`
3. **Kiểm tra còn bao nhiêu quota**: `python3 tools/free_quota.py --used kaggle=2`

### Nhắc lại giới hạn của Kaggle (đừng thiết kế pipeline phụ thuộc vào nó)

| Giới hạn | Hệ quả |
|---|---|
| ~30 h GPU/tuần | Đủ để dev, không đủ để train full |
| Không SSH, không chạy nền | Không thể dùng làm worker cho `compute_dire.py` |
| Máy bị xoá khi hết phiên | Mọi thứ quan trọng phải ghi vào `/kaggle/working` rồi tải về |
| P100 16 GB | Không dùng được batch lớn / mixed precision mới |
| ToS | Không dùng thủ thuật đa tài khoản để lách quota |